In [ ]:
from graphviz import Digraph
import numpy as np
import pandas as pd
from datetime import datetime, timedelta


file_path = "bases/para tesis LCC_v3.xlsx"
df_totales = pd.read_excel(file_path)

df_planes_por_alumno = pd.read_excel(file_path, sheet_name="planes por alumno")
df_cursadas_por_alumno = pd.read_excel(file_path, sheet_name="cursadas por alumno (2)")
df_lista_de_materias = pd.read_excel(file_path, sheet_name="lista de materias")
df_finales_por_alumno = pd.read_excel(file_path, sheet_name="finales por alumno (2)") #sheet_name="finales por alumno")
df_estados_de_actas = pd.read_excel(file_path, sheet_name="estados de actas")
df_resultados_posibles = pd.read_excel(file_path, sheet_name="resultados posibles")

In [ ]:
year = 2016

In [ ]:
# cursaronYEAR = df_cursadas_por_alumno[(df_cursadas_por_alumno['anio_academico'] == year) 
#                                      & (df_cursadas_por_alumno['periodo'].isin(['1er Cuatrimestre', '1er Bimestre'])) 
#                                      & (df_cursadas_por_alumno['codigo_materia'].isin(['COMP930001', 'COMP930007']))
#                                      & (df_cursadas_por_alumno['persona'].isin(df_planes_por_alumno['persona']))  ][['persona']].drop_duplicates('persona').copy() #personaID que cursaron en YEAR
# # print(len(cursaron2022))  

# preYEAR = df_cursadas_por_alumno[df_cursadas_por_alumno['anio_academico'] < year][['persona']].drop_duplicates('persona') #personasID con inscripcion  pre-YEAR
# camadaYEAR = cursaronYEAR.drop( cursaronYEAR[cursaronYEAR['persona'].isin( list(preYEAR.transpose().iloc[0].values) )].index ) #camada2022 = cursaron2022 SIN los que se han inscripto pre-YEAR
# camadaYEAR

In [ ]:
yearList = [year]

personaXyear = df_cursadas_por_alumno[ #( df_cursadas_por_alumno['codigo_materia'].isin(['COMP930001', 'COMP930007'])) &
                (df_cursadas_por_alumno['persona'].isin(df_planes_por_alumno['persona'])) ].groupby('persona')['anio_academico'].min().reset_index()
# camada[camada['anio_academico'].isin(yearList)]
camadaYEAR = personaXyear[personaXyear['anio_academico'].isin(yearList) ]
camadaYEAR

In [ ]:
dic_materias = df_lista_de_materias[df_lista_de_materias['codigo_materia'].str.contains('COMP93')].sort_values(by="codigo_materia")[:17].drop_duplicates().reset_index().drop(columns=('index'))
dic_materias

In [ ]:
#todos los finales de los estudiantes de la 
# - camada de year, 
# - finales aprobados,
# - y es de las materias obligatorias

finales_camadaYEAR = df_finales_por_alumno[(df_finales_por_alumno['resultado'] == 'A') & 
                                           (df_finales_por_alumno['persona'].isin(camadaYEAR['persona'].to_list())) & 
                                           (df_finales_por_alumno['codigo_materia'].isin(dic_materias['codigo_materia'].tolist()) )]
finales_camadaYEAR

In [ ]:
# finales_camadaYEAR[ finales_camadaYEAR['persona'] =  ]

In [ ]:
#agrego vector con todos ceros
finales_camadaYEAR.insert(len(finales_camadaYEAR.columns), 'vector', [[0]*len(dic_materias) for i in range(len(finales_camadaYEAR)) ]  )
finales_camadaYEAR = finales_camadaYEAR.sort_values(['persona', 'fecha_cierre_acta' ] ) # hago sort por persona, luego year, luego periodo
finales_camadaYEAR

##### recorrer a los finales en ese orden

In [ ]:
for filaNro in range(1, len(finales_camadaYEAR)):
    oldFila = finales_camadaYEAR.iloc[filaNro-1]
    fila = finales_camadaYEAR.iloc[filaNro]

    if int(oldFila.persona) == int(fila.persona):
        finales_camadaYEAR.at[int(fila.name), 'vector']  = oldFila.vector

    if fila.resultado == 'A':
        newVect = finales_camadaYEAR.loc[int(fila.name), 'vector'].copy()
        k = int(dic_materias[dic_materias['codigo_materia'] == fila.codigo_materia ]['codigo_materia'].index[0])
        newVect[k] = 1
        finales_camadaYEAR.at[int(fila.name), 'vector'] = newVect

finales_camadaYEAR

In [ ]:
finales_camadaYEAR['vectStr'] = finales_camadaYEAR['vector'].apply(str) # agrego copia de fila 'vector' pero como string
# finales_camadaYEAR

#### Segunda pasada

In [ ]:
finales_camadaYEAR[['vectStr', 'vector']].copy().drop_duplicates('vectStr').reset_index().drop(columns=('index')) 

In [ ]:
vectorCeroStandar = str([0]*len(dic_materias))
edges = []
persona = []
esLoop = []

#asocia nodo <-> numero
dic_node = finales_camadaYEAR[['vectStr', 'vector']].copy().drop_duplicates('vectStr').reset_index().drop(columns=('index')) # index es el nombre del nodo que usaré para nodos de grafic
if(len(dic_node[dic_node['vectStr'] == vectorCeroStandar]) == 0):
    dic_node.loc[-1] = [vectorCeroStandar, [0]*len(dic_materias) ]
    dic_node.index = dic_node.index + 1

for p in finales_camadaYEAR['persona'].drop_duplicates().tolist():
    personaCursadas = finales_camadaYEAR[finales_camadaYEAR['persona'] == p]

    vectA = vectorCeroStandar

    oldFila = personaCursadas.iloc[0]
    oldDate = oldFila.fecha_cierre_acta

    
    for i in range(0,len(personaCursadas)): 
        fila = personaCursadas.iloc[i]
        laDate = fila.fecha_cierre_acta

        if oldDate!= laDate: 
            originPoint = int(dic_node[dic_node['vectStr'] == vectA].iloc[0].name) 
            endPoint = int(dic_node[dic_node['vectStr'] == oldFila.vectStr ].iloc[0].name)

            edges.append((originPoint, endPoint))
            persona.append(int(fila.persona))
            esLoop.append(originPoint==endPoint)

            vectA = oldFila.vectStr


        oldFila = fila
        oldDate = laDate

        isAproved = fila.resultado


    if vectA!=oldFila.vectStr and isAproved=='A':
        originPoint = int(dic_node[dic_node['vectStr'] == vectA].iloc[0].name) 
        endPoint = int(dic_node[dic_node['vectStr'] == oldFila.vectStr ].iloc[0].name)
    
        edges.append((originPoint, endPoint))
        persona.append(int(fila.persona))
        esLoop.append(originPoint==endPoint)
    
        
# edges

In [ ]:
allEdges = pd.DataFrame({'edges' : edges, 'persona': persona, 'esLoop': esLoop})

#Loops 
losLoops = allEdges[allEdges['esLoop']].drop_duplicates().copy()
losEdges = allEdges[~(allEdges['esLoop'])]

allAllEdges = pd.concat([losEdges,losLoops])


df_edges = pd.DataFrame({'edge': allAllEdges['edges'].drop_duplicates()  })
df_edges.insert(1, 'appears', [ len(allAllEdges[allAllEdges['edges'] == df_edges['edge'].iloc[i]]) for i in range(len(df_edges)) ] ) 
df_edges.insert(2, 'idList', [allAllEdges[allAllEdges['edges'] == df_edges['edge'].iloc[i] ]['persona'].to_list() for i in range(len(df_edges)) ])
df_edges

### Hacer grafico

In [ ]:
materiasCatalogo = pd.merge(dic_materias, df_lista_de_materias, on='codigo_materia', how='left')
materiasCatalogo['codigo'] = ['alg','algo1','algo2','algo3','analisis', 'bd', 'inge1','inge2', 'LyC','metNum','orga1','orga2','plp','proba','SO','redes', 'tleng']

def strMateriasAprobadas(l): #l <- nodosCaminos.iloc[i]['vector']
    boolean_mask = [bool(valor) for valor in l]
    # temp= pd.merge(dic_materias[boolean_mask], df_lista_de_materias, on='codigo_materia', how='left')
    output = f""
    for i in range(len(materiasCatalogo[boolean_mask])): #  materiasCatalogo[boolean_mask]['codigo'].tolist():
        output+= str(materiasCatalogo[boolean_mask].iloc[i].name+1) +'. '+ materiasCatalogo[boolean_mask].iloc[i].codigo +'\n'
    return output[:len(output)-1]


##### sugerencia de Mati: limitar a nodos con aristas de >= 3 personas

In [ ]:
df_edges = df_edges[df_edges['appears'] >= 2 ]

nodosDelPlan = [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 
                [1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #alg y analisis
                [1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0], #alg, algo1. analisis. proba'
                [1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0], #alg. algo1. algo2. analisis. orga1. proba'
                [1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0], #alg. algo1. algo2. analisis. metNum. orga1. orga2. proba'
                [1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0], #alg. algo1. algo2. algo3. analisis. metNum. orga1. orga2. proba. SO'
                [1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0], #alg. algo1. algo2. algo3. analisis. inge1. LyC. metNum. orga1. orga2. proba. SO. redes
                [1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1], #alg. algo1. algo2. algo3. analisis. bd. inge1. LyC. metNum. orga1. orga2. plp. proba. SO. redes. tleng'
                [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]  #alg. algo1. algo2. algo3. analisis. bd. inge1. inge2. LyC. metNum. orga1. orga2. plp. proba. SO. redes. tleng'
                ]

In [ ]:
tpl = set()
for tuple in df_edges['edge'].tolist():
    tpl.add(tuple[0])
    tpl.add(tuple[1])
#tpl son los index de los nodos que SI les llega o sale una edge de ellos
tpl = list(tpl)

In [ ]:
dot = Digraph()
for i in tpl: #range(len(dic_node)):
    nombre =  strMateriasAprobadas(dic_node.loc[i]['vector'])

    if(dic_node.loc[i]['vector'] in nodosDelPlan):
        elColor="#62b3ff"
    else:
        elColor="#000000"
    
    dot.node(str(i), str(i) +'\n' + nombre, color = elColor)

for i in range(len(df_edges)):
    beginning = df_edges['edge'].iloc[i][0]
    end = df_edges['edge'].iloc[i][1]

    width = df_edges['appears'].iloc[i]

    if((dic_node.loc[beginning]['vector'] in nodosDelPlan) and 
       (dic_node.loc[end]['vector'] in nodosDelPlan) and 
       abs(nodosDelPlan.index(dic_node.loc[beginning]['vector']) - nodosDelPlan.index(dic_node.loc[end]['vector'])) == 1):
        elColor="#62b3ff"
    else:
        elColor="#5c5c5c"
    dot.edge(str(beginning), str(end), penwidth = str( width/4 ), label=str( width ), color=elColor)

dot

___________________________________
# Calidad de Datos

### Distancia con Practico y Final  
fecha_init_cuatri_aprox: es una medida aproximada de la fecha INICIAL de cursada. Duda: donde estan los ANEXOS de las "Ordenes del dia"?

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
df_calendario = pd.read_csv("bases/output_fechasCalendarioAcademico.csv")
df_calendario.sort_values("Anio Academico", ascending=False) #Nombre, Periodo Generico, Fecha Inicio Dict, Fecha Fin Dict

In [ ]:
df_calendario.rename(columns={"Nombre": "periodo_lectivo", "Anio Academico": "anio_academico", "Fecha Fin Dict": "Fecha_Fin_Dict" }, inplace=True)
df_calendario['Fecha_Fin_Dict'] = pd.to_datetime(df_calendario['Fecha_Fin_Dict'], format='%Y-%m-%d')

In [ ]:
# df_calendario['periodo'] = df_calendario['periodo'].str.replace(".", "")
# df_calendario['periodo'] = df_calendario['periodo'].str.replace(" 2025", "")
# df_calendario['periodo'] = df_calendario['periodo'].str.replace(" 2024", "")
# df_calendario['periodo'] = df_calendario['periodo'].str.replace(" 2023", "")
# df_calendario['periodo'] = df_calendario['periodo'].str.replace(" 2022", "")
# df_calendario['periodo'] = df_calendario['periodo'].str.replace(" 2021", "")
# df_calendario['periodo'] = df_calendario['periodo'].str.replace(" 2020", "")

#### dic_materias tiene es las 17 obligatorias del plan93

- Elijo personas de las materias obligatorias, y aprobadas

In [ ]:
### Elijo personas de las materias obligatorias, y aprobadas                   comment: ese YEAR camada, en
cursadas = df_cursadas_por_alumno[      #df_cursadas_por_alumno['persona'].isin(camadaYEAR['persona'].tolist()) & 
                       (df_cursadas_por_alumno['codigo_materia'].isin(dic_materias['codigo_materia'].tolist())) &
                       (df_cursadas_por_alumno['resultado'] == 'A')]

#cursadas['fecha_init_cuatri_aprox'] = [ datetime.strptime(str(cursadas.loc[i].anio_academico)+"-"+('03' if cursadas.loc[i].periodo == '1er Cuatrimestre' else ( '01' if cursadas.loc[i].periodo == '1er Bimestre' else '08'))+('-20' if cursadas.loc[i].periodo == '1er Bimestre' else "-15"), "%Y-%m-%d") for i in cursadas.index]
#cursadas['fecha_fin_cuatri'] = 
#cursadas

In [ ]:
# cursadas[cursadas['anio_academico']==2014  ]['periodo'].value_counts()
# df_calendario[df_calendario['anio_academico']==2014 ][['periodo', 'anio_academico', 'Fecha_Fin_Dict']]

- Agrego a las cursadas, la informacion de fecha de finDeDictada

In [ ]:
cursadas = pd.merge(cursadas, df_calendario[['periodo_lectivo', 'anio_academico', 'Fecha_Fin_Dict']], on=['periodo_lectivo','anio_academico'], how='left')
cursadas

- Elijo finales de:
i. las materias obligatorias, y 
ii. aprobadas 
iii. en instancia regular
iv. estado Acta Cerrrada

In [ ]:
finales = df_finales_por_alumno[                #df_finales_por_alumno['persona'].isin(camadaYEAR['persona'].tolist()) & 
                      (df_finales_por_alumno['codigo_materia'].isin(dic_materias['codigo_materia'].tolist())) & 
                      (df_finales_por_alumno['resultado'] == 'A') &
                      (df_finales_por_alumno['instancia'] == 3)&
                      (df_finales_por_alumno['estado_acta'] == 'C')]
finales

*DUDA 29/06*: hay finales APROBADOS con acta CERRADA duplicados para estos estudiantes, que procede? los saco?

In [ ]:
finales

In [ ]:
finales[finales[['persona', 'codigo_materia']].duplicated()] #finales[finales['persona']==30637]

- Agrego a las cursadas, la info de su finalAprobado correspondiente para la misma persona y materia

In [ ]:
cursadas_y_finales = pd.merge(cursadas, finales, on=['persona', 'codigo_materia'], how='left')
cursadas_y_finales.sort_values(['persona', 'anio_academico', 'periodo' ])

In [ ]:
# "Aprox cantidad de dias en un cuatrimestre: " + str(timedelta(days=7*16))[:8]
# cursadas_y_finales = cursadas_y_finales[~cursadas_y_finales['Fecha_Fin_Dict'].isna() ]


#### CalidadDeDatos: fecha_cierre_acta este bien (proxima al cuatri cuando se impartió la materia)

In [ ]:
# cursadas_y_finales['dist_cierrerActa_y_finCuatriAprox'] = [(cursadas_y_finales.iloc[i].fecha_cierre_acta_x - (cursadas_y_finales.iloc[i].fecha_init_cuatri_aprox + (timedelta(days=7*7) if cursadas_y_finales.iloc[i].periodo == '1er Bimestre' else timedelta(days=7*16) ))   ) for i in cursadas_y_finales.index]

cursadas_y_finales['dist_cierrerActa_y_finCuatriAprox'] = [(cursadas_y_finales.iloc[i].fecha_cierre_acta_x - cursadas_y_finales.iloc[i].Fecha_Fin_Dict) for i in cursadas_y_finales.index]
cursadas_y_finales[['persona', 'codigo_materia', 'periodo', 'anio_academico', 'Fecha_Fin_Dict', 'fecha_cierre_acta_x',  'dist_cierrerActa_y_finCuatriAprox']]

In [ ]:
avg_dist_cierrerActa_y_finCuatriAprox = cursadas_y_finales.groupby(['codigo_materia'])
df_calidadDatosActa = pd.merge(dic_materias, avg_dist_cierrerActa_y_finCuatriAprox['dist_cierrerActa_y_finCuatriAprox'].median(), on='codigo_materia', how='left')
df_calidadDatosActa

In [ ]:
# temp = cursadas_y_finales[['codigo_materia', 'dist_cierrerActa_y_finCuatriAprox']]
temp=  pd.merge(dic_materias, cursadas_y_finales, on='codigo_materia', how='left')
temp

In [ ]:
temp['dias_cierre'] = temp['dist_cierrerActa_y_finCuatriAprox'].dt.days

In [ ]:
# Configuramos el tamaño del gráfico
plt.figure(figsize=(12, 4))

# Eliminamos los valores nulos (NaN) para que no falle el gráfico
datos = temp['dist_cierrerActa_y_finCuatriAprox'].dropna().dt.days

# # Creamos el boxplot (vert=False lo hace horizontal, suele ser más fácil de leer)
# plt.boxplot(datos, vert=False, patch_artist=True, 
#             # boxprops=dict(facecolor='lightblue', color='blue'),
#             medianprops=dict(color='blue', linewidth=2))

# Creamos el boxplot (vert=False lo hace horizontal, suele ser más fácil de leer)
sns.boxplot(data=temp[temp['dias_cierre'] <= 1000], 
            x='dias_cierre',      # Los datos numéricos en el eje horizontal
            color='skyblue',      # Color de las cajas
            medianprops={"color": "b", "linewidth": 2},
            flierprops={"marker": "x"},
            showfliers=True )

# Personalización del gráfico
plt.title('Distancia entre Cierre de Acta Cursada y Fin de Cuatrimestre', fontsize=14)
plt.xlabel('Días entre Cierre de Acta y Fin del Cuatrimestre', fontsize=10)
plt.yticks([0], ['Todas las materias'])
plt.grid(axis='x', linestyle='--', alpha=0.7)

# Línea vertical en 0 (idealmente el cierre debería ser cerca de este valor)
# plt.axvline(x=0, color='red', linestyle='--', label='Cierre ideal (0 días)')

# Línea de referencia: El ideal es que se cierren al mes de haber terminadoa de dictar la materia (100% arbitrario este numero)
plt.axvline(x=30, color='red', linestyle='--', label='Al mes del Fin del cuatrimestre (Día 30)')

plt.legend()

plt.show()

In [ ]:
print(len(temp[temp['dias_cierre'] > 800]))
print(len(temp[temp['dias_cierre'] <= 800]))

In [ ]:
len(temp[temp['dias_cierre'] > 900])*100 / len(temp[temp['dias_cierre'] <= 800])

In [ ]:
temp[temp['dias_cierre'] > 800][['codigo_materia','periodo','anio_academico']].drop_duplicates()

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

In [ ]:
plt.figure(figsize=(12, 8))
font = 13

# Seaborn hace la magia: agrupa por 'y' y calcula la distribución de 'x'
sns.boxplot(
    data=temp,#[temp['dias_cierre'] <= 800],  ############duda: puedo cortar los outliers en 1000? mas de eso es muchisimo 
    x='dias_cierre',      # Los datos numéricos en el eje horizontal
    y='nombre_materia',   # Las categorías en el eje vertical
    color='skyblue',      # Color de las cajas
    medianprops={"color": "b", "linewidth": 2},
    flierprops={"marker": "x"},
    showfliers=True       # Muestra a los estudiantes "atípicos" como puntos
)

# 4. PERSONALIZACIÓN
#plt.title('Mediana de Días de Retraso en Cierre de Actas por Materia', fontsize=18)
plt.xlabel('Días de diferencia entre Cierre de Acta Cursada y Fin del Cuatrimestre', fontsize=font)
plt.ylabel('Materia', fontsize=font)

# # Línea de referencia: El ideal es que se cierren al mes de haber terminadoa de dictar la materia (100% arbitrario este numero)
plt.axvline(x=30, color='red', linestyle='--', label='Al mes del Fin del cuatrimestre (Día 30)')

plt.grid(axis='x', linestyle='--', alpha=0.7)


plt.xticks(fontsize=font)
plt.yticks(fontsize=font, rotation=45)

# legend_elements = [
#     mpatches.Patch(facecolor='lightblue', edgecolor='black',
#                    label='Rango intercuartílico (P25–P75)'),
#     Line2D([0], [0], color='blue', lw=2,
#            label='Mediana'),
#     Line2D([0], [0], color='black', lw=1,
#            marker='|', markersize=10, linestyle='-',
#            label='Rango sin outliers (mín.–máx.)'),
#     Line2D([0], [0], marker='x', color='gray', linestyle='None',
#            markersize=7, label='Valores atípicos (outliers)'),
#     Line2D([0], [0], color='red', lw=1.5, linestyle='--',
#            label='Umbral de referencia (Día 30)'),
# ]

# plt.legend(
#     handles=legend_elements,
#     loc='lower right',
#     fontsize=8,
#     framealpha=0.9,
#     title='Referencias'
# )

# plt.legend()
plt.tight_layout()

In [ ]:
plt.figure(figsize=(12, 8))
font = 13

# Seaborn hace la magia: agrupa por 'y' y calcula la distribución de 'x'
sns.boxplot(
    data=temp[temp['dias_cierre'] <= 800],  ############duda: puedo cortar los outliers en 1000? mas de eso es muchisimo 
    x='dias_cierre',      # Los datos numéricos en el eje horizontal
    y='nombre_materia',   # Las categorías en el eje vertical
    color='skyblue',      # Color de las cajas
    medianprops={"color": "b", "linewidth": 2},
    flierprops={"marker": "x"},
    showfliers=True       # Muestra a los estudiantes "atípicos" como puntos
)

# 4. PERSONALIZACIÓN
#plt.title('Mediana de Días de Retraso en Cierre de Actas por Materia', fontsize=18)
plt.xlabel('Días de diferencia entre Cierre de Acta Cursada y Fin del Cuatrimestre', fontsize=font)
plt.ylabel('Materia', fontsize=font)

# # Línea de referencia: El ideal es que se cierren al mes de haber terminadoa de dictar la materia (100% arbitrario este numero)
plt.axvline(x=30, color='red', linestyle='--', label='Al mes del Fin del cuatrimestre (Día 30)')

plt.grid(axis='x', linestyle='--', alpha=0.7)


plt.xticks(fontsize=font)
plt.yticks(fontsize=font, rotation=45)

# legend_elements = [
#     mpatches.Patch(facecolor='lightblue', edgecolor='black',
#                    label='Rango intercuartílico (P25–P75)'),
#     Line2D([0], [0], color='blue', lw=2,
#            label='Mediana'),
#     Line2D([0], [0], color='black', lw=1,
#            marker='|', markersize=10, linestyle='-',
#            label='Rango sin outliers (mín.–máx.)'),
#     Line2D([0], [0], marker='x', color='gray', linestyle='None',
#            markersize=7, label='Valores atípicos (outliers)'),
#     Line2D([0], [0], color='red', lw=1.5, linestyle='--',
#            label='Umbral de referencia (Día 30)'),
# ]

# plt.legend(
#     handles=legend_elements,
#     loc='lower right',
#     fontsize=8,
#     framealpha=0.9,
#     title='Referencias'
# )

# plt.legend()
plt.tight_layout()

#### para AVG distancia TP vs Final

In [ ]:
cursadas_y_finales['dist_tp_al_final'] = [(cursadas_y_finales.iloc[i].fecha_llamado_mesa - cursadas_y_finales.iloc[i].Fecha_Fin_Dict  ) for i in cursadas_y_finales.index] #- timedelta(days=7*16)
cursadas_y_finales[['persona', 'codigo_materia', 'periodo', 'anio_academico', 'Fecha_Fin_Dict', 'fecha_llamado_mesa', 'dist_tp_al_final' ]]

### Ranking NO colgar el final por materia

In [ ]:
median_dist_tp_al_final = cursadas_y_finales.groupby('codigo_materia')
df_calidadDatos = pd.merge(dic_materias, median_dist_tp_al_final['dist_tp_al_final'].median() , on='codigo_materia', how='left')
df_calidadDatos

__________________________________________________
### Grafo c/estudiantes vs tiempoHastaDarElFinal

In [ ]:
materia='COMP930019' #tleng

In [ ]:
df_tiempo_final_vs_cursada = cursadas_y_finales[cursadas_y_finales['codigo_materia'] == materia ].sort_values(by=['Fecha_Fin_Dict', 'dist_tp_al_final']) 

In [ ]:
print(len(df_tiempo_final_vs_cursada)) ## gente que curso y: no rindio final y/o no aprobo
df_tiempo_final_vs_cursada = df_tiempo_final_vs_cursada[~df_tiempo_final_vs_cursada['fecha_llamado_mesa'].isna() ]
len(df_tiempo_final_vs_cursada)

https://stackoverflow.com/questions/59747313/how-can-i-plot-a-confidence-interval-in-python

In [ ]:
plt.rcParams["font.size"] =7
plt.figure(figsize=(130, 100))

data_dict = {}
data_dict['persona'] = df_tiempo_final_vs_cursada['persona'].tolist()
data_dict['Fecha_Fin_Dict'] = df_tiempo_final_vs_cursada['Fecha_Fin_Dict']  # [ x for x in range(len(df_tiempo_final_vs_cursada[:3])) ]
data_dict['fecha_llamado_mesa'] = df_tiempo_final_vs_cursada['fecha_llamado_mesa'] #[x+1 for x in range(len(df_tiempo_final_vs_cursada))]
dataset = pd.DataFrame(data_dict)

for lower,upper,y in zip(dataset['Fecha_Fin_Dict'],dataset['fecha_llamado_mesa'],range(len(dataset))):
    plt.plot((lower,upper),(y,y),'o-',color='orange')
plt.yticks(range(len(dataset)),list(dataset['persona']))
plt.show()

In [ ]:

cursadas_y_finales[(~cursadas_y_finales['fecha_llamado_mesa'].isna()) & 
                   (cursadas_y_finales['codigo_materia'] == 'COMP930001')  & 
                   (cursadas_y_finales['anio_academico'] == 2013)  & 
                   (cursadas_y_finales['periodo'] == '1er Bimestre') ]


In [ ]:
def filtrado_materia(df):
    df.sort_values(by=['dist_tp_al_final'], inplace=True, ascending=True)
    n = df.shape[0] // 2
    nMenos1 = max(0,n-1)
    nMas1 = min(n, df.shape[0]-1)

    minimio = 0
    maximo = df.shape[0]-1

    df = df.iloc[[minimio, nMenos1, n, nMas1, maximo]]
    df.drop_duplicates(inplace=True)
    return df

def filtrado_materia_cuartos(df):
    df.sort_values(by=['dist_tp_al_final'], inplace=True, ascending=True)
    n25 = df.shape[0] // 4
    n50 = df.shape[0] // 2
    n75 = (df.shape[0] *3) // 4

    primerCuarto = max(0,n25)

    mitad = max(n25,n50)

    ultimoCuarto = min(n75, df.shape[0]-1)

    minimio = 0
    maximo = df.shape[0]-1

    df = df.iloc[[primerCuarto, mitad, ultimoCuarto]] #minimio,maximo
    df.drop_duplicates(inplace=True)
    return df   

In [ ]:
cursadas_y_finales_group = cursadas_y_finales[ (~cursadas_y_finales['fecha_llamado_mesa'].isna())  ].groupby(['codigo_materia', 'anio_academico','periodo']).apply(lambda x: filtrado_materia_cuartos(x)) 
cursadas_y_finales_group 

In [ ]:
# cursadas_y_finales[~cursadas_y_finales['fecha_llamado_mesa'].isna() ].groupby(['codigo_materia', 'anio_academico','periodo'])[['fecha_llamado_mesa','Fecha_Fin_Dict']].median()

In [ ]:
# pd.merge(dic_materias, cursadas_y_finales[['codigo_materia', 'anio_academico','periodo']], on='codigo_materia', how='left'  )

In [ ]:
# cursadas_y_finales_group = cursadas_y_finales[~cursadas_y_finales['fecha_llamado_mesa'].isna() ].groupby(['codigo_materia', 'anio_academico','periodo'])

# medianaDeMateria = pd.merge(pd.merge(dic_materias, cursadas_y_finales[['codigo_materia', 'anio_academico','periodo']].drop_duplicates(), on='codigo_materia', how='left'  ), 
#          cursadas_y_finales_group['fecha_llamado_mesa'].median() , 
#          on=['codigo_materia', 'anio_academico','periodo'],  
#          how='left')

# medianaDeMateria = pd.merge(medianaDeMateria, 
#          cursadas_y_finales_group['Fecha_Fin_Dict'].median() , 
#          on=['codigo_materia', 'anio_academico','periodo'],  
#          how='left')

# medianaDeMateria = pd.merge(medianaDeMateria, 
#          cursadas_y_finales_group['persona'].median() , 
#          on=['codigo_materia', 'anio_academico','periodo'],  
#          how='left')

# medianaDeMateria

In [ ]:
cursadas_y_finales_group

In [ ]:
plt.rcParams["font.size"] =5
plt.figure(figsize=(18, 9))

# df_cursadaVSfinal = medianaDeMateria[(medianaDeMateria['codigo_materia']==materia) & ~(medianaDeMateria['fecha_llamado_mesa'].isna() )  ] #SOLO CON MEDIANA
df_cursadaVSfinal = cursadas_y_finales_group[cursadas_y_finales_group['codigo_materia'] == 'COMP930001'] #MEDIANA EN CUARTOS

data_dict = {}
data_dict['persona'] = df_cursadaVSfinal['persona'].tolist()
data_dict['Fecha_Fin_Dict'] = df_cursadaVSfinal['Fecha_Fin_Dict']  # [ x for x in range(len(df_tiempo_final_vs_cursada[:3])) ]
data_dict['fecha_llamado_mesa'] = df_cursadaVSfinal['fecha_llamado_mesa'] #[x+1 for x in range(len(df_tiempo_final_vs_cursada))]
dataset = pd.DataFrame(data_dict)

for lower,upper,y in zip(dataset['Fecha_Fin_Dict'],dataset['fecha_llamado_mesa'],range(len(dataset))):
    plt.plot((lower,upper),(y,y),'o-',color='orange')

years = dataset['fecha_llamado_mesa'].dt.year.unique()
plt.xticks(
    ticks=pd.to_datetime([f"{y}-01-01" for y in years]),
    labels=years
)

plt.yticks(range(len(dataset)),list(dataset['persona']))
plt.grid(True,axis='x', linestyle='--', linewidth=0.5)
plt.show()

In [ ]:
len(dic_materias['codigo_materia'].tolist())

In [ ]:
cursadas_y_finales_group[(cursadas_y_finales_group['codigo_materia'] == 'COMP930003') & (cursadas_y_finales_group['anio_academico']==2020) ]

In [ ]:
fig, axs = plt.subplots(4, 5, figsize=(20, 16), layout='constrained')

for ax, mat in zip(axs.flat, range(len(dic_materias))) :
    materia = dic_materias.iloc[mat]['codigo_materia']
    df_cursadaVSfinal = cursadas_y_finales_group[(cursadas_y_finales_group['codigo_materia']==materia)  ] #& ~(medianaDeMateria['fecha_llamado_mesa'].isna() ) 

    data_dict = {}
    data_dict['persona'] = df_cursadaVSfinal['persona'].tolist()
    data_dict['Fecha_Fin_Dict'] = df_cursadaVSfinal['Fecha_Fin_Dict']  # [ x for x in range(len(df_tiempo_final_vs_cursada[:3])) ]
    data_dict['fecha_llamado_mesa'] = df_cursadaVSfinal['fecha_llamado_mesa'] #[x+1 for x in range(len(df_tiempo_final_vs_cursada))]
    dataset = pd.DataFrame(data_dict)

    ax.set_title(f'Materia={dic_materias.iloc[mat]['nombre_materia']}')
    for lower,upper,y in zip(dataset['Fecha_Fin_Dict'],dataset['fecha_llamado_mesa'],range(len(dataset))):
        ax.plot((lower,upper),(y,y),'o-',
                color='orange',
            markersize=2.5,
            linewidth=0.8,
            alpha=0.75)

    years = dataset['Fecha_Fin_Dict'].dt.year.unique()
    
    ax.set_xticks(
        ticks=pd.to_datetime([f"{y}-01-01" for y in years]),
        labels=years
    )

    if(materia == 'COMP930003'):
        print(years)
        print(pd.to_datetime([f"{y}-01-01" for y in years]))

    ax.set_yticks(range(len(dataset)),list(dataset['persona']))
    ax.grid(True,axis='x', linestyle='--', linewidth=0.5)




##para quitar los cuadros extras al final
total_subplots = len(axs.flat)
materias_graficadas = len(dic_materias)

# Ocultamos los subplots que no se utilizaron
for i in range(materias_graficadas, total_subplots):
    axs.flat[i].set_visible(False)

fig.suptitle('Tiempo entre finalizar la Cursada vs aprobación Final por Materia', fontsize=18, y=1.02)

## Attempt nuevo grafico cursada aprobada vs finalRendido:
#### para anexo

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ── Figura completa: 3 filas × 5 columnas, todas las materias ──────────────
fig, axs = plt.subplots(6, 3, figsize=(20, 38), layout='constrained')

for ax, mat in zip(axs.flat, range(len(dic_materias))):
    materia = dic_materias.iloc[mat]['codigo_materia']
    nombre  = dic_materias.iloc[mat]['nombre_materia']

    df_m = cursadas_y_finales_group[
        cursadas_y_finales_group['codigo_materia'] == materia
    ].copy().reset_index(drop=True)

    for idx, row in df_m.iterrows():
        ax.plot(
            [row['Fecha_Fin_Dict'], row['fecha_llamado_mesa']],
            [idx, idx],
            'o-',
            color='orange',
            markersize=4,
            linewidth=1.6,
            alpha=0.75
        )

    # Ejes X: solo años, rotados
    ax.xaxis.set_major_locator(mdates.YearLocator(1))   # cada 1 años
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.tick_params(axis='x', labelsize=15, rotation=45)

    # Eje Y: sin etiquetas de persona (muy densas para PDF)
    ax.set_yticks([])
    ax.set_ylabel('Estudiantes', fontsize=15, labelpad=3)

    # Título del subgráfico: nombre corto, dos líneas si hace falta
    # nombre_corto = nombre.replace('Algoritmos y Estructuras de Datos',
    #                                'AED').replace('Organización del Computador',
    #                                'Org. Comp.')
    ax.set_title(nombre, fontsize=16, pad=3, wrap=True)
    ax.grid(True, axis='x', linestyle='--', linewidth=0.4, alpha=0.6)
    ax.spines[['top', 'right']].set_visible(False)

# Ocultar subplots sobrantes
for i in range(len(dic_materias), len(axs.flat)):
    axs.flat[i].set_visible(False)

# fig.suptitle('Tiempo entre Aprobación de Cursada y Examen Final, por Materia',
#                 fontsize=18)

# fig.savefig('anexo_scatter_todas_materias.pdf', bbox_inches='tight', dpi=200, backend='pdf')
plt.show()

#### para algunas en especifico

In [ ]:
materiasCatalogo

In [ ]:
MATERIAS_REPR = {
    'COMP930003': 'Algoritmos y Estructuras de Datos I',
    'COMP930004': 'Algoritmos y Estructuras de Datos II',
    'COMP930015': 'Sistemas Operativos',
    'COMP930008': 'Base de datos', #
}

fig, axs = plt.subplots(2, 2, figsize=(12, 10), layout='constrained')
# fig.subplots_adjust(wspace=0.35, hspace=0.35)

for ax, (codigo, label) in zip(axs.flat, MATERIAS_REPR.items()):
    df_m = cursadas_y_finales_group[
        cursadas_y_finales_group['codigo_materia'] == codigo
    ].copy().reset_index(drop=True)

    for idx, row in df_m.iterrows():
        ax.plot(
            [row['Fecha_Fin_Dict'], row['fecha_llamado_mesa']],
            [idx, idx],
            'o-',
            color='#E8920A',
            markersize=3,
            linewidth=1.0,
            alpha=0.8
        )

    ax.xaxis.set_major_locator(mdates.YearLocator(1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.tick_params(axis='x', labelsize=10, rotation=45)
    ax.set_yticks([])
    ax.set_ylabel('Estudiantes', fontsize=font)
    ax.set_title(label, fontsize=font, pad=6) #, fontweight='bold'
    ax.grid(True, axis='x', linestyle='--', linewidth=0.5, alpha=0.6)
    ax.spines[['top', 'right']].set_visible(False)

# fig.suptitle('Tiempo entre Aprobación de Cursada y Examen Final (Materias Representativas)', fontsize=12, y=1.03)

fig.savefig('fig_scatter_repr.pdf', bbox_inches='tight', dpi=200)
plt.show()